# Init

In [0]:
# importing liabraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import *
from pyspark.sql import Window

# Rename Config

In [0]:
# Rename config
RENAME_MAP = {
    "CID": "cid",
    "BDATE": "birth_date",
    "GEN": "gender"
}

# Read from Bronze

In [0]:
# read spark table
df = spark.table("workspace.bronze.erp_cust_az12_raw")

# Transformation

## Rename column

In [0]:
# column Rename
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Extracting customer_key

In [0]:
from pyspark.sql import functions as F

df = df.withColumn(
    "customer_key",
    F.when(
        F.col("cid").startswith("NAS"), 
        # Starts at position 4 and grabs text until the end of the string
        F.substring(F.col("cid"), 4, F.length(F.col("cid")))
    ).otherwise(F.col("cid"))
)


## Eliminate invalid birth_date

In [0]:
# ==============================================================================
# DATA QUALITY RULE: REMOVE FUTURE BIRTHDATES
# Logic: If a birthdate is in the future compared to today, clear it to NULL.
# ==============================================================================
df = df.withColumn(
    "birth_date",
    F.when(F.col("birth_date") > F.current_date(), F.lit(None))
     .otherwise(F.col("birth_date"))
)


## Trim

In [0]:
# string data triming
for field in df.schema.fields:
    if field.dataType == StringType():
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.display()

# Write in Silver

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.erp_cust_az12")
)